In [1]:
# Notebook copy of NLPManager (same logic as your nlp_manager.py)

import re
import faiss
import torch
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


class NLPManager:
    loaded = False

    def __init__(self):
        # EMBEDDING MODEL
        self.embedding_model = SentenceTransformer("models/bge-base-en-v1.5")

        # RERANKER
        self.reranker = CrossEncoder("models/bge-reranker-base")

        # TEXT SPLITTER
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=700,
            chunk_overlap=150,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        # LLM
        self.MODEL_NAME = "models/Qwen2.5-1.5B-Instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )

        # CONFIDENCE GATE
        self.CONF_L4_GATE = 0.25

        # STORAGE
        self.chunks = []
        self.chunk_to_doc = []
        self.chunk_embeddings = None
        self.faiss_index = None
        self.bm25 = None
        self._token_re = re.compile(r"[a-zA-Z0-9]+")

    def bm25_tokenize(self, text: str) -> list[str]:
        return self._token_re.findall(text.lower())

    def retrieval_confidence(self, reranked: list[dict]) -> float:
        if not reranked:
            return 0.0
        score = float(reranked[0].get("rerank_score", 0.0))
        return max(0.0, min(1.0, score))

    def load_corpus(self, documents: list[dict[str, str]]) -> None:
        chunks = []
        chunk_to_doc = []

        for doc in documents:
            doc_id = doc["id"]
            text = doc["document"]

            split_chunks = self.text_splitter.split_text(text)
            for chunk in split_chunks:
                chunks.append(chunk)
                chunk_to_doc.append(doc_id)

        self.chunks = chunks
        self.chunk_to_doc = chunk_to_doc

        # EMBEDDINGS
        embeddings = self.embedding_model.encode(
            chunks,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype("float32")
        self.chunk_embeddings = embeddings

        # FAISS
        dim = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)
        self.faiss_index.add(embeddings)

        # BM25
        tokenized_chunks = [self.bm25_tokenize(c) for c in chunks]
        self.bm25 = BM25Okapi(tokenized_chunks)

        self.loaded = True

    def hybrid_retrieve(
        self,
        question: str,
        dense_k: int = 20,
        sparse_k: int = 20,
        top_k: int = 8,
        alpha: float = 0.65
    ) -> list[dict]:

        q_emb = self.embedding_model.encode(
            [question],
            normalize_embeddings=True
        ).astype("float32")

        D, I = self.faiss_index.search(q_emb, dense_k)

        dense_scores = {
            int(idx): float(score)
            for score, idx in zip(D[0], I[0])
            if idx >= 0
        }

        q_tok = self.bm25_tokenize(question)
        bm25_scores = self.bm25.get_scores(q_tok)

        if sparse_k >= len(bm25_scores):
            top_sparse_idx = np.argsort(bm25_scores)[::-1]
        else:
            top_sparse_idx = np.argpartition(bm25_scores, -sparse_k)[-sparse_k:]

        sparse_scores = {int(i): float(bm25_scores[i]) for i in top_sparse_idx}

        candidates = sorted(set(dense_scores.keys()) | set(sparse_scores.keys()))
        if not candidates:
            return []

        svals = np.array([sparse_scores.get(i, 0.0) for i in candidates], dtype="float32")
        smin = float(svals.min())
        smax = float(svals.max())

        def normalize_sparse(v: float) -> float:
            return (v - smin) / (smax - smin + 1e-9)

        combined = []
        for idx in candidates:
            dense = dense_scores.get(idx, 0.0)
            sparse = normalize_sparse(sparse_scores.get(idx, 0.0))
            score = alpha * dense + (1 - alpha) * sparse

            combined.append({
                "chunk": self.chunks[idx],
                "doc_id": self.chunk_to_doc[idx],
                "score": float(score)
            })

        combined.sort(key=lambda x: x["score"], reverse=True)
        return combined[:top_k]

    def rerank(self, question: str, retrieved: list[dict], top_k: int = 2) -> list[dict]:
        if not retrieved:
            return []

        pairs = [[question, r["chunk"]] for r in retrieved]
        scores = self.reranker.predict(pairs)

        out = []
        for r, s in zip(retrieved, scores):
            out.append({**r, "rerank_score": float(s)})

        out.sort(key=lambda x: x["rerank_score"], reverse=True)
        return out[:top_k]

    def build_prompt(self, question: str, contexts: list[str]) -> str:
        context_text = "\n\n".join(contexts)
        prompt = f"""
You are a question answering system.

Answer ONLY using the provided context.

If the answer is not found in the context,
say EXACTLY:
NOT_FOUND

Context:
{context_text}

Question:
{question}

Answer:
"""
        return prompt

    def qa(self, question: str) -> dict[str, list[str] | str]:
        retrieved = self.hybrid_retrieve(
            question,
            dense_k=8,
            sparse_k=8,
            top_k=4
        )

        if not retrieved:
            return {"documents": [], "answer": ""}

        # FAST PATH
        top_score = retrieved[0]["score"]
        if top_score > 0.85:
            reranked = retrieved[:2]
            for r in reranked:
                r["rerank_score"] = r["score"]
        else:
            reranked = self.rerank(question, retrieved, top_k=2)

        # DOCUMENT IDS
        documents = []
        for r in reranked:
            if r["doc_id"] not in documents:
                documents.append(r["doc_id"])
        documents = documents[:3]

        # CONFIDENCE CHECK (L4)
        conf = self.retrieval_confidence(reranked)
        if conf < self.CONF_L4_GATE:
            return {"documents": [], "answer": ""}

        # CONTEXTS -> GENERATE
        contexts = [r["chunk"] for r in reranked]
        prompt = self.build_prompt(question, contexts)

        output = self.generator(
            prompt,
            max_new_tokens=32,
            do_sample=False,
            temperature=0.1,
            return_full_text=False
        )
        answer = output[0]["generated_text"].strip()

        # L5
        if answer.upper().startswith("NOT_FOUND"):
            return {"documents": documents, "answer": ""}

        return {"documents": documents, "answer": answer}

In [2]:
from collections import defaultdict

def _reciprocal_rank(pred_docs, gold_docs, k):
    gold = set(gold_docs)
    for i, d in enumerate(pred_docs[:k], start=1):
        if d in gold:
            return 1.0 / i
    return 0.0

def _recall_at_k(pred_docs, gold_docs, k):
    gold = set(gold_docs)
    return 1.0 if any(d in gold for d in pred_docs[:k]) else 0.0

def _precision_at_k(pred_docs, gold_docs, k):
    gold = set(gold_docs)
    top = pred_docs[:k]
    if not top:
        return 0.0
    rel = sum(1 for d in top if d in gold)
    return rel / float(k)

def evaluate(instances, preds, ks=(1,3,5)):
    assert len(instances) == len(preds)

    by_diff = defaultdict(list)
    for i, ex in enumerate(instances):
        by_diff[ex.get("difficulty","UNK")].append(i)

    results = {}

    def eval_answerable(indices, name):
        rows = []
        for k in ks:
            mrr = rec = prec = 0.0
            n = 0
            for i in indices:
                ex = instances[i]
                if not ex.get("answerable", False):
                    continue
                gold_docs = ex.get("source_docs", [])
                if not gold_docs:
                    continue
                pred_docs = preds[i].get("documents", []) or []
                mrr += _reciprocal_rank(pred_docs, gold_docs, k)
                rec += _recall_at_k(pred_docs, gold_docs, k)
                prec += _precision_at_k(pred_docs, gold_docs, k)
                n += 1
            rows.append((k, None, None, None, 0) if n == 0 else (k, mrr/n, rec/n, prec/n, n))
        results[name] = rows

    def eval_unanswerable(indices, name):
        n_l4 = n_l5 = ok_l4 = ok_l5 = 0
        for i in indices:
            ex = instances[i]
            if ex.get("answerable", True):
                continue
            diff = ex.get("difficulty","")
            pred_docs = preds[i].get("documents", []) or []
            if diff == "L4":
                n_l4 += 1
                if len(pred_docs) == 0:
                    ok_l4 += 1
            if diff == "L5":
                n_l5 += 1
                if len(pred_docs) > 0:
                    ok_l5 += 1
        results[name] = {
            "L4_doc_empty_acc": (ok_l4 / n_l4) if n_l4 else None,
            "L4_count": n_l4,
            "L5_doc_nonempty_acc": (ok_l5 / n_l5) if n_l5 else None,
            "L5_count": n_l5,
        }

    # overall + by diff
    eval_answerable(list(range(len(instances))), "answerable_all")
    for diff in sorted(by_diff.keys()):
        eval_answerable(by_diff[diff], f"answerable_{diff}")

    eval_unanswerable(list(range(len(instances))), "unanswerable_format_all")
    return results

def pretty_print(results):
    for key in results:
        if key.startswith("answerable_"):
            print(f"\n=== {key} ===")
            for (k, mrr, rec, prec, n) in results[key]:
                if n == 0:
                    print(f"k={k}: n=0")
                else:
                    print(f"k={k}: n={n:4d} | MRR@{k}={mrr:.4f} | Recall@{k}={rec:.4f} | Precision@{k}={prec:.4f}")

    u = results["unanswerable_format_all"]
    print("\n=== unanswerable_format_all ===")
    print(f"L4: n={u['L4_count']} | doc-empty acc={u['L4_doc_empty_acc']}")
    print(f"L5: n={u['L5_count']} | doc-nonempty acc={u['L5_doc_nonempty_acc']}")

results = evaluate(instances_eval, preds, ks=(1,3,5))
pretty_print(results)

NameError: name 'instances_eval' is not defined

In [ ]:
# show a few retrieval misses for answerable questions
misses = []
for ex, pr in zip(instances_eval, preds):
    if not ex.get("answerable", False):
        continue
    gold = set(ex.get("source_docs", []))
    pred = set(pr.get("documents", []) or [])
    if not gold.intersection(pred):
        misses.append((ex.get("difficulty"), ex.get("key"), ex["question"], list(gold), list(pred)))

print("Answerable retrieval misses:", len(misses))
for row in misses[:15]:
    diff, key, q, gold, pred = row
    print("\n---")
    print("diff:", diff, "key:", key)
    print("Q:", q)
    print("Gold:", gold)
    print("Pred:", pred)

# L5 formatting failures (should return docs but you returned none)
l5_fail = []
for ex, pr in zip(instances_eval, preds):
    if ex.get("difficulty") == "L5":
        if len(pr.get("documents", []) or []) == 0:
            l5_fail.append((ex.get("key"), ex["question"]))
print("\nL5 doc-nonempty failures:", len(l5_fail))
for key, q in l5_fail[:10]:
    print("\nkey:", key)
    print("Q:", q)